# DISCUS_SUITE from Python: command catalogue, macros and runs

This notebook demonstrates the catalogue of DISCUS_SUITE commands that discuspdf ships with, and the tools built
on it for writing and running macros for any program of the suite (DISCUS, KUPLOT, REFINE, DIFFEV, MIXSCAT):

1. **The command catalogue.** Every command each menu of DISCUS_SUITE 6.23.01 accepts, read from the program's
   source. It answers "does this command exist here, and which menu does it open?" and resolves abbreviations the
   way DISCUS does.
2. **The macro checker.** Follows a macro line by line and says which menu each line is read in, so a misspelt
   command is caught before DISCUS sees it.
3. **The macro builder.** Writes macros from Python, with menus as `with` blocks, and refuses lines DISCUS would
   read differently from what the code intends.
4. **Running a macro.** Runs any macro in its own folder and records the macro, the inputs and their checksums,
   so every run can be traced later.
5. **Coverage.** Which commands already have a typed Python API.

Sections 1–3 and 5 need only Python. **Section 4 runs DISCUS** (three short runs, a few seconds each).

In [ ]:
from pathlib import Path
import os, sys, json, platform, tempfile
sys.path.insert(0, str(Path.cwd().parent))
os.chdir(Path.cwd().parent)                     # repository root, like the other notebooks

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 72
from discuspdf.suite_catalogue import load_catalogue
from discuspdf.macro import Macro, MacroError
from discuspdf.engine import Engine
from discuspdf.suite_runner import SuiteRunner

GAAS = Path("tests/fixtures/discus_6_23_01/gaas.cll")

# Where DISCUS runs. On Windows, DISCUS_SUITE lives in WSL; elsewhere it is on the PATH (or named by
# DISCUSPDF_ENGINE). Run folders must have no spaces in their path, so they go to C:\tmp on Windows.
if platform.system() == "Windows":
    ENGINE = Engine(("wsl.exe", "-d", os.environ.get("DISCUS_WSL_DISTRO", "Ubuntu-22.04"), "--",
                     "discus_suite", "-noautorun", "-macro"), timeout_seconds=600)
    WORK = Path(os.environ.get("DISCUSPDF_NB_WORK", r"C:\tmp\nb_suite")) / "nb11"
else:
    ENGINE = Engine.from_env(timeout_seconds=600)
    WORK = Path(os.environ.get("DISCUSPDF_NB_WORK", tempfile.mkdtemp(prefix="nb_suite_"))) / "nb11"
print("DISCUS_SUITE is started with:", " ".join(ENGINE.launcher))

## 1. The command catalogue

DISCUS_SUITE is driven by text commands, typed at a prompt or listed in a macro file. Each program (DISCUS,
KUPLOT, …) has a main menu and sub-menus: in DISCUS, `fourier` opens the Fourier menu, where `set`, `run` and so
on mean something else than in the main menu, and `exit` returns.

The catalogue was built by reading the Fortran routines that decide which command a typed word is, and the help
files for the syntax. It covers 44 menus.

In [ ]:
# load_catalogue(version="6.23.01") — the bundled command catalogue (built from DiffuseCode tag v.6.23.01)
#   returns   Catalogue: .menus {path: Menu}, .engine (version, commit), .n_commands
cat = load_catalogue()
print(cat)
programs = {}
for path, menu in cat.menus.items():
    programs.setdefault(menu.program, []).append(path)
for prog, paths in programs.items():
    print(f"{prog:8s} {len(paths):2d} menus:", ", ".join(p.split("/", 1)[-1] for p in paths[:12]),
          "…" if len(paths) > 12 else "")

**One menu in detail.** The commands of the DISCUS Fourier menu, with the shortest abbreviation DISCUS accepts
for each, and the help text's syntax for `set`. The catalogue says what DISCUS *accepts*, not what a command
does physically; only running it shows that.

In [ ]:
# Catalogue.menu(path) — one menu: .commands (Command: name, min_abbrev, enters, optional, keywords, help)
# Catalogue.shortest_abbreviation(command) — the shortest prefix DISCUS resolves to that command
#   (it can be longer than the minimum in the source when an earlier branch shadows it)
four = cat.menu("discus/fourier")
rows = [(c.name, cat.shortest_abbreviation(c)) for c in four.commands if not c.standard]
for i in range(0, len(rows), 5):
    print("   ".join(f"{name:10s} [{abbr}]" for name, abbr in rows[i:i + 5]))
print("\n'set' in the Fourier menu, from the help file:")
for line in four.command("set").syntax[:6]:
    print("  ", line)
print("its name:value options:", four.command("set").optional)

**How DISCUS reads a typed word.** DISCUS compares only the first few characters, and the first matching command
in the source wins. Some consequences, all taken from the source and resolved here by `Catalogue.resolve`:

* `four` in DISCUS is `fourier`, and `t` in the Fourier menu is already `temp`.
* KUPLOT keeps only 4 characters of every command word, so `mergeg` runs `merge`.
* In the KUPLOT fit menu, `con` runs `continue`, so `co` is the shortest way to type `convergence`.

In [ ]:
# Catalogue.resolve(menu, word) — the Command DISCUS runs for `word` typed in `menu` (None if unknown)
for menu, word in [("discus", "four"), ("discus/fourier", "t"), ("kuplot", "mergeg"),
                   ("kuplot/fit", "con"), ("kuplot/fit", "co"), ("discus", "fourir")]:
    c = cat.resolve(menu, word)
    target = f"{c.name}" + (f"  (opens {c.enters})" if c.enters else "") if c else "unknown command"
    print(f"{menu:15s} {word!r:9s} -> {target}")

## 2. The macro checker

`walk` follows a macro the way DISCUS reads it: line by line, keeping track of the menu it is in. Here is a
hand-written macro with one typo (`fourir`). DISCUS stops a macro at the first error, so a typo found only at
run time can waste a long calculation; the checker finds it at once.

Two more details show up in the table: the `read` menu takes exactly one line and returns by itself (no `exit`),
and a line with `=` is a variable assignment, not a command.

In [ ]:
text = '''discus
read
  cell gaas.cll,2,2,2
fourir
fourier
  set aver,100
  n = 3
  run
exit
save supercell.stru
exit
'''
# Catalogue.walk(text, start="suite") — one Step per macro line as DISCUS reads it
#   Step: line, text, menu (the menu the line is read in), kind ("command", "unknown", "assignment",
#         "comment", "control", "macro", "after_end"), command (resolved full name)
for s in cat.walk(text):
    print(f"{s.line:2d}  {s.text:22s} read in {s.menu:15s} {s.kind:10s} {s.command or ''}")
# Catalogue.check(text) — only the lines DISCUS would reject as unknown (empty list = all known)
print("\nproblems:", [(s.line, s.text) for s in cat.check(text)])

## 3. The macro builder

Instead of writing macro text by hand, `Macro` builds it from Python. Menus are `with` blocks, which write the
`exit` themselves. Numbers are formatted once, the same way everywhere (integers as they are, floats with up to
12 significant digits). This macro reads the 2-atom GaAs unit cell as a 2×2×2 crystal and saves it.

In [ ]:
# Macro(title=None) — a DISCUS_SUITE macro starting at the suite level
#   m.program(name)          with-block: enter a program (discus, kuplot, refine, diffev, mixscat)
#   m.menu(word, *params)    with-block: enter a sub-menu; writes 'exit' at the end (read needs none)
#   m.command(word, *params, **options)   one line 'word p1,p2,name:value'
#   m.raw(line)              a line written as given (assignments, do/if blocks, @macro calls)
#   m.render()               the macro text, after checking every line against the catalogue
m = Macro("GaAs 2x2x2 supercell")
with m.program("discus"):
    with m.menu("read"):
        m.command("cell", "gaas.cll", 2, 2, 2)
    m.command("save", "supercell.stru")      # 'save <file>' writes at once; a bare 'save' would open a menu
print(m.render())

**What it refuses.** Each of these would make DISCUS do something else than intended: an unknown command, a
command that opens a menu written as a single line (the following lines would land in the sub-menu), and a
second line inside the one-line `read` menu (it would be read in the DISCUS main menu).

In [ ]:
def attempt(label, build):
    try:
        build()
        print(f"{label:34s} accepted")
    except MacroError as exc:
        print(f"{label:34s} refused: {exc}")

def typo():
    m = Macro()
    with m.program("discus"):
        m.command("fourir")

def menu_as_line():
    m = Macro()
    with m.program("discus"):
        m.command("fourier")

def two_lines_in_read():
    m = Macro()
    with m.program("discus"):
        with m.menu("read"):
            m.command("cell", "gaas.cll", 1, 1, 1)
            m.command("cell", "gaas.cll", 2, 2, 2)

attempt("misspelt command", typo)
attempt("menu command written as a line", menu_as_line)
attempt("two lines in the read menu", two_lines_in_read)

The builder checks names and menus, not what the arguments mean. A command without a typed discuspdf API can
still be written with `command`, or with `raw` for things such as variable assignments; raw lines are checked
against the catalogue too.

In [ ]:
m = Macro("a Fourier layer, written with the general builder")
with m.program("discus"):
    with m.menu("read"):
        m.command("cell", "gaas.cll", 4, 4, 4)
    m.raw("n = 51")                               # a DISCUS variable assignment
    with m.menu("fourier"):
        m.command("xray")
        m.command("ll", -4, -4, 0); m.command("lr", 4, -4, 0); m.command("ul", -4, 4, 0)
        m.command("nabs", "n"); m.command("nord", "n")
        m.command("set", "aver", 0)
        m.command("run")
print(m.render())

## 4. Running a macro

`SuiteRunner` runs any macro. `prepare` creates a fresh folder with the macro (`run.mac`), copies of the inputs
under the names the macro uses, and `run.json` with the DISCUS_SUITE command and the SHA-256 checksum of every
file. Nothing has
run yet, so the folder can be inspected first. `execute` checks the checksums again, runs DISCUS, and writes
`status.json`. A run is marked `complete` only if DISCUS exited cleanly, printed no error, and wrote every declared
output file.

`complete` means DISCUS ran cleanly and wrote the files, not that the results are physically right.

In [ ]:
runner = SuiteRunner(WORK / "runs", ENGINE)
# SuiteRunner.prepare(macro, inputs, outputs) — stage a run without starting DISCUS
#   macro     Macro or text;  inputs  {name used in the macro: file};  outputs  files the macro must write
#   returns   SuiteRun (.directory, .macro, .record = run.json, .status(), .output(name))
m = Macro("GaAs 2x2x2 supercell")
with m.program("discus"):
    with m.menu("read"):
        m.command("cell", "gaas.cll", 2, 2, 2)
    m.command("save", "supercell.stru")
run = runner.prepare(m, {"gaas.cll": GAAS}, ["supercell.stru"])
print("files staged:", sorted(p.name for p in run.directory.iterdir()))
print(run.macro.read_text())
rec = run.record
print("run.json: started with", " ".join(rec["engine"]["launcher"]), "| inputs", rec["inputs"], "| outputs", rec["outputs"])
for name, digest in rec["sha256"].items():
    print(f"  sha256 {name:10s} {digest[:16]}…")

In [ ]:
# SuiteRunner.execute(run) — re-check the hashes, run DISCUS, check the log and outputs, write status.json
runner.execute(run)
print(json.dumps(run.status(), indent=1))

The saved crystal is read back as a discuspdf `Structure`. DISCUS `save` writes no crystal size, so the size
(2×2×2 cells) is given when reading. GaAs has 8 atoms per cubic cell, so 64 atoms are expected.

In [ ]:
from discuspdf import read_stru
from discuspdf.structure_viz import plot_structure

# read_stru(path, supercell=None) — read a DISCUS .stru as a Structure (cell-major order is checked)
#   supercell   crystal size in unit cells; needed because DISCUS 'save' writes no ncell line
sc = read_stru(run.output("supercell.stru"), supercell=(2, 2, 2)).structure
print(sc.n_atoms, "atoms;", {s: int((sc.species == s).sum()) for s in sorted(set(sc.species))})
fig = plt.figure(figsize=(5, 5))
plot_structure(sc, ax=fig.add_subplot(projection="3d"));

**Not only DISCUS.** The same runner drives KUPLOT, the suite's plotting and fitting program. Here KUPLOT tabulates
sin(x) from 0 to 6 in steps of 0.5 and saves it with the `ksave` menu (which returns to KUPLOT by itself after
`run`, so the builder writes no `exit` for it).

In [ ]:
k = Macro("KUPLOT: tabulate a function and save it")
with k.program("kuplot"):
    k.command("function", "sin(r[0])", 0, 6, 0.5)
    with k.menu("ksave", 1):
        k.command("outfile", "sine.xy")
        k.command("format", "xy")
        k.command("run")
print(k.render())
krun = runner.run(k, (), ["sine.xy"])           # prepare + execute in one call
print("status:", krun.status()["status"])
xy = np.loadtxt(krun.output("sine.xy"))
print(f"{len(xy)} points; largest difference from numpy sin: {np.abs(xy[:, 1] - np.sin(xy[:, 0])).max():.1e}")
fig, ax = plt.subplots(figsize=(5, 3))
xx = np.linspace(0, 6, 200)
ax.plot(xx, np.sin(xx), color="0.7", label="numpy sin(x)")
ax.plot(*xy.T, "o", label="KUPLOT, saved by ksave")
ax.set(xlabel="x", ylabel="y"); ax.legend();

**A failed run.** The macro reads a file that does not exist. DISCUS prints an error, and the runner marks the run
`failed` with the reason, instead of returning an empty or stale result. The folder keeps the log for inspection.

In [ ]:
bad = Macro("must fail: missing input file")
with bad.program("discus"):
    with bad.menu("read"):
        bad.command("cell", "does_not_exist.cll", 1, 1, 1)
    bad.command("save", "never.stru")
brun = runner.prepare(bad, (), ["never.stru"])
try:
    runner.execute(brun)
except RuntimeError as exc:
    print("RuntimeError:", str(exc).split(" (see")[0])
print(json.dumps(brun.status(), indent=1))

## 5. Coverage: which commands have a typed API

Some commands are written for you by discuspdf's typed tools (structure operations, Fourier calculations, powder
PDFs, Monte Carlo); this chart counts them per menu. Every other command is still usable through
`Macro.command` and `Macro.raw`, as in section 3, and is checked against the catalogue in the same way.

In [ ]:
import warnings
sys.path.insert(0, "tools/suite_catalogue")
from coverage import used_commands            # walks macros through the catalogue and collects their commands
from render_samples import sample_macros      # one macro from each of discuspdf's typed tools

with warnings.catch_warnings():
    warnings.simplefilter("ignore")           # some sample settings print warnings that do not matter here
    generated = used_commands(cat, [text for _label, text in sample_macros()])
rows = []
for path, menu in cat.menus.items():
    names = [c.name for c in menu.commands if not c.standard]
    if names and (menu.program == "discus" or path in ("kuplot", "refine", "diffev", "mixscat")):
        rows.append((path, len(names), len(set(names) & generated.get(path, set()))))
rows.sort(key=lambda r: -r[1])
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(rows))
ax.bar(x, [r[1] for r in rows], color="0.85", label="commands in the menu")
ax.bar(x, [r[2] for r in rows], width=0.5, label="written by a typed discuspdf tool")
ax.set_xticks(x, [r[0].replace("discus/", "") for r in rows], rotation=70, fontsize=8)
ax.set_ylabel("commands"); ax.legend(); fig.tight_layout()
print("typed commands:", sum(r[2] for r in rows), "of", sum(r[1] for r in rows))

**What you see.** Grey: how many commands each menu has. The DISCUS Fourier and waves menus have a typed API for
most of their commands, the DISCUS main menu for about a fifth (reading, saving, and the structure operations of
notebook 12); KUPLOT, REFINE, DIFFEV and most DISCUS sub-menus are used through the general builder.

## Things to keep in mind

* **The catalogue is about syntax.** A macro that passes the checker can still compute the wrong thing; only
  running it, and checking the result, shows that.
* **Run folders need paths without spaces** (DISCUS splits file names at blanks). On Windows they go to `C:\tmp`.
* **A finished run is never overwritten.** `execute` only accepts a run in the `prepared` state; to run again,
  prepare a new one.